<a href="https://colab.research.google.com/github/harshith-murali/BIS-Lab/blob/main/Ant%20Colony%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import math


def aco_pcb(grid, nets, ants=40, iterations=100,
            alpha=1, beta=3, rho=0.2, Q=100,
            bend_penalty=2, seed=42):

    rng = random.Random(seed)
    rows, cols = len(grid), len(grid[0])
    pheromone = {}

    best_routing = None
    best_cost = math.inf

    # Undirected edge: both directions share pheromone.
    def edge(a, b):
        return tuple(sorted((a, b)))

    def neighbors(cell):
        r, c = cell
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                yield nr, nc

    terminals = {point for net in nets for point in net}

    for _ in range(iterations):
        complete_solutions = []

        for _ in range(ants):
            occupied = set()
            routing = {}
            total_cost = 0

            order = list(range(len(nets)))
            rng.shuffle(order)

            for net_id in order:
                source, destination = nets[net_id]
                current = source
                path = [source]
                visited = {source}
                direction = None
                path_cost = 0

                other_terminals = terminals - {source, destination}

                while current != destination:
                    candidates = []
                    weights = []

                    for nxt in neighbors(current):
                        r, c = nxt

                        if (grid[r][c] == 1 or nxt in occupied
                                or nxt in visited
                                or nxt in other_terminals):
                            continue

                        new_direction = (
                            nxt[0] - current[0],
                            nxt[1] - current[1]
                        )

                        turn = (
                            direction is not None
                            and new_direction != direction
                        )

                        distance = (
                            abs(r - destination[0])
                            + abs(c - destination[1])
                        )

                        heuristic = 1 / (
                            1 + distance + bend_penalty * turn
                        )

                        key = (net_id, edge(current, nxt))
                        tau = pheromone.get(key, 1.0)

                        candidates.append(nxt)
                        weights.append(tau**alpha * heuristic**beta)

                    if not candidates:
                        break  # Dead end: discard this ant's attempt.

                    nxt = rng.choices(candidates, weights=weights)[0]
                    new_direction = (
                        nxt[0] - current[0],
                        nxt[1] - current[1]
                    )

                    turn = (
                        direction is not None
                        and new_direction != direction
                    )

                    path_cost += 1 + bend_penalty * turn
                    direction = new_direction
                    current = nxt
                    path.append(current)
                    visited.add(current)

                if current != destination:
                    break

                routing[net_id] = path
                occupied.update(path)
                total_cost += path_cost

            if len(routing) == len(nets):
                complete_solutions.append((routing, total_cost))

                if total_cost < best_cost:
                    best_routing = routing
                    best_cost = total_cost

        # Evaporate existing pheromones.
        for key in pheromone:
            pheromone[key] = max(
                0.01, (1 - rho) * pheromone[key]
            )

        # Reinforce edges belonging to complete solutions.
        for routing, cost in complete_solutions:
            deposit = Q / max(cost, 1)

            for net_id, path in routing.items():
                for a, b in zip(path, path[1:]):
                    key = (net_id, edge(a, b))
                    pheromone[key] = (
                        pheromone.get(key, 1.0) + deposit
                    )

    return best_routing, best_cost


# 0 = free cell, 1 = obstacle
grid = [
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0]
]

# Each net contains (source, destination), using (row, column).
nets = [
    ((0, 0), (5, 0)),
    ((0, 5), (5, 5))
]

routing, cost = aco_pcb(grid, nets)

if routing is None:
    print("No complete routing found.")
else:
    print("Best routing cost:", cost)

    for net_id in sorted(routing):
        print(f"Net {net_id + 1}: {routing[net_id]}")

Best routing cost: 10
Net 1: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0)]
Net 2: [(0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
